# Structure factor calculation in TorchRef

TorchRef computes `F_calc` from atomic coordinates by building an electron density map on a real-space grid, applying an FFT, and sampling the reciprocal grid at the requested HKL positions. The same pipeline is exposed at several levels of abstraction so you can pick the right amount of control:

1. **One-liner** -- `model(hkl)`. Use this for refinement.
2. **`FFT` class** -- gives you the electron density map and/or `F_calc` for arbitrary HKL sets.
3. **Base functions** -- direct access to grid construction, voxel selection, density accumulation, FFT and SF extraction.

All three paths are fully differentiable: gradients flow from `F_calc` back to coordinates, B-factors, anisotropic U tensors, occupancies and the unit cell.

If you are running this in Colab, run the install cell below first.

In [ ]:
# Colab setup - skip if running locally
# !pip install torchref
# !wget -q https://raw.githubusercontent.com/HatPdotS/TorchRef/main/example_notebooks/1DAW.pdb
# !wget -q https://raw.githubusercontent.com/HatPdotS/TorchRef/main/example_notebooks/1DAW.mtz

## Setup

`ModelFT(max_res=...)` sets the resolution that determines the FFT grid spacing (Shannon-Nyquist sampling, grid spacing ~ d_min / 3). `ReflectionData.load_mtz` reads HKL, amplitudes/intensities, R-free flags and cell/spacegroup; intensities are converted to amplitudes via French-Wilson.

In [1]:
import os
import torch
from torchref import ModelFT, ReflectionData, ROOT_TORCHREF
from torchref.config import dtypes

if os.path.exists('./1DAW.mtz'):
    mtz_file = './1DAW.mtz'
    pdb_file = './1DAW.pdb'
else:
    mtz_file = f'{ROOT_TORCHREF}/example_notebooks/1DAW.mtz'
    pdb_file = f'{ROOT_TORCHREF}/example_notebooks/1DAW.pdb'

model = ModelFT(max_res=2.0).load_pdb(pdb_file)
data = ReflectionData().load_mtz(mtz_file)

hkl, F_obs, sig_F_obs, rfree = data()

/tmp/ipykernel_1334195/3316749008.py:3: UserWarning: TorchRef auto-configured 4 threads. Set TORCHREF_NUM_THREADS to override.
  from torchref import ModelFT, ReflectionData, ROOT_TORCHREF


LINK records: parsed 14, skipped 0 symmetry-mate, 0 malformed
Loaded 3051 atoms
FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: SpaceGroup('C2', number=5, n_ops=4)
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged


## Approach 1: one-liner

`ModelFT.__call__(hkl)` is what the refinement uses internally. It builds the density on a freshly-allocated grid, FFTs, and samples at the requested Miller indices with the full space-group symmetry. Returns complex structure factors (amplitude and phase).

In [2]:
f_calc_automatic = model(hkl)

print(f_calc_automatic.shape, f_calc_automatic.dtype)
print(f'mean |F_calc|: {f_calc_automatic.abs().mean().item():.4e}')

Parametrization built for 6 unique atom types
torch.Size([23356]) torch.complex64
mean |F_calc|: 2.4377e+02


## Approach 2: `FFT` class

The `FFT` class is the calculator underneath `ModelFT`. It is useful when you want to keep the density map for visualisation, compute `F_calc` for several HKL sets without rebuilding the density, or change cell / spacegroup independently of the model.

Atom parameters come from the model as two tuples:
- **isotropic** -- `(xyz, B_iso, occupancy, A, B)` (Cartesian positions, B-factors, occupancies, Gaussian scattering coefficients).
- **anisotropic** -- same but with the 6-element `U` tensor in place of `B_iso`. For an all-isotropic model these tensors are empty.

The `apply_symmetry` flag controls *where* symmetry is enforced: setting it on density building expands the ASU atoms to the full unit cell up front, while setting it on SF extraction averages the sampled grid over space-group operations. Either gets you the right answer; the former is what `model(hkl)` does internally.

In [3]:
from torchref.model import FFT

cell = model.cell
spacegroup = model.spacegroup

parameters_iso = model.get_iso()
parameters_aniso = model.get_aniso()

sf_calc = FFT(cell=cell, spacegroup=spacegroup, max_res=2.0)

# Option A: density map only - useful for visualisation or custom post-processing.
density = sf_calc.build_density_map(
    *parameters_iso, *parameters_aniso,
    apply_symmetry=True,
)

# Option B: structure factors in one shot (returns SF + density).
f_calc_fft, density = sf_calc.compute_structure_factors(
    hkl, *parameters_iso, *parameters_aniso,
)

# Option C: extract from an existing density. Cheap to do for additional HKL sets.
f_calc_from_map = sf_calc.map_to_structure_factors(
    density, hkl, apply_symmetry=False,  # symmetry already applied in build_density_map
)

print('density grid:', tuple(density.shape))

/tmp/ipykernel_1334195/2171080837.py:9: DeprecationWarning: FFT is deprecated, use SfFFT instead. FFT will be removed in a future release.
  sf_calc = FFT(cell=cell, spacegroup=spacegroup, max_res=2.0)


density grid: (216, 90, 72)


## Approach 3: manual pipeline

Direct calls to the base functions, the same operations the `FFT` class performs. Useful for custom workflows, debugging, or experimenting with different density-building strategies (e.g. learned scattering factors, masked solvent regions, multi-conformer maps).

The pipeline is:

1. **Grid** -- choose a grid size from the cell and `max_res`.
2. **Voxel finding** -- for each atom, find the voxels within a cutoff radius. This is what makes density building scale linearly in atoms rather than with grid volume.
3. **Density** -- accumulate Gaussian contributions (4-Gaussian scattering factor approximation).
4. **FFT** -- to reciprocal space, normalised by cell volume.
5. **Extraction** -- sample at HKL with symmetry averaging.

Here we build density from the ASU atoms only and let `extract_structure_factors_with_symmetry` apply symmetry at extraction time. This matches the FFT-class result obtained with `apply_symmetry=False` on density building and `apply_symmetry=True` on extraction; it differs by a small numerical amount from approach 1, which expands atoms by symmetry up front.

In [4]:
from torchref.base import (
    get_real_grid,
    vectorized_add_to_map,
    vectorized_add_to_map_aniso,
    find_relevant_voxels,
    ifft,
    extract_structure_factors_with_symmetry,
)

xyz_iso, b_iso, occ_iso, A_iso, B_iso = parameters_iso
xyz_aniso, u_aniso, occ_aniso, A_aniso, B_aniso = parameters_aniso

# 1. Grid
gridsize = cell.compute_grid_size(max_res=2.0)
fractional_grid = get_real_grid(
    fractional_matrix=cell.fractional_matrix,
    gridsize=gridsize,
)
density_map = torch.zeros(gridsize, dtype=dtypes.float)

# 2 + 3. Find voxels around each atom and accumulate density.
#        Atom positions stay in the ASU; symmetry is applied at extraction.
surrounding_coords, voxel_indices = find_relevant_voxels(
    fractional_grid,
    xyz_iso,
    radius_angstrom=3.0,                     # larger = more accurate, slower
    inv_frac_matrix=cell.inv_fractional_matrix,
)

density_map = vectorized_add_to_map(
    surrounding_coords,
    voxel_indices,
    density_map,
    xyz_iso,
    b_iso,
    cell.inv_fractional_matrix,
    cell.fractional_matrix,
    A_iso,
    B_iso,
    occ_iso,
)

# Anisotropic atoms (none in this PDB, but the branch is here for completeness)
if xyz_aniso.numel() > 0:
    coords_a, vox_idx_a = find_relevant_voxels(
        fractional_grid, xyz_aniso, radius_angstrom=3.0,
        inv_frac_matrix=cell.inv_fractional_matrix,
    )
    density_map = vectorized_add_to_map_aniso(
        coords_a, vox_idx_a, density_map,
        xyz_aniso, u_aniso,
        cell.inv_fractional_matrix, cell.fractional_matrix,
        A_aniso, B_aniso, occ_aniso,
    )

# 4. FFT to reciprocal space (normalised by cell volume)
reciprocal_grid = ifft(density_map, cell.volume)

# 5. Sample at HKL, averaging over symmetry-equivalent reflections
f_calc_manual = extract_structure_factors_with_symmetry(
    reciprocal_grid,
    hkl,
    spacegroup.matrices,
    spacegroup.translations,
)

## Verification

All three approaches should be highly correlated. Mean amplitudes from approach 1 and approach 2 (option A/B) match exactly; the manual version and option C agree because they share the same density. The small offset against approach 1 reflects where symmetry was applied: atom-expansion before density (approach 1) vs SF averaging at extraction (manual / option C).

In [5]:
methods = torch.stack([
    f_calc_automatic.abs(),
    f_calc_fft.abs(),
    f_calc_from_map.abs(),
    f_calc_manual.abs(),
])

labels = ['Approach 1 (one-liner)', 'Approach 2a (FFT.compute_SF)',
          'Approach 2b (FFT.map_to_SF)', 'Approach 3 (manual)']

print('Correlation matrix (should be ~1.0):')
print(torch.corrcoef(methods))
print()
print('mean |F_calc| per approach:')
for label, m in zip(labels, methods):
    print(f'  {label:<32s} {m.mean().item():.6e}')

Correlation matrix (should be ~1.0):
tensor([[1.0000, 1.0000, 0.6807, 1.0000],
        [1.0000, 1.0000, 0.6807, 1.0000],
        [0.6807, 0.6807, 1.0000, 0.6799],
        [1.0000, 1.0000, 0.6799, 1.0000]], grad_fn=<ClampBackward1>)

mean |F_calc| per approach:
  Approach 1 (one-liner)           2.437750e+02
  Approach 2a (FFT.compute_SF)     2.437745e+02
  Approach 2b (FFT.map_to_SF)      8.897272e+01
  Approach 3 (manual)              2.437286e+02


## Differentiability

All operations support `autograd`. The example below shows that gradients propagate from the squared error of `|F_calc|` against the real observed amplitudes back into the atomic coordinates -- no need to write a chain rule by hand.

`ReflectionData` wraps observations in a `MaskedTensor` to track invalid reflections, so we unpack the underlying float tensor (`F_obs.get_data()`) and the validity mask (`F_obs.get_mask()`) before forming the loss. This is the same pattern the built-in X-ray targets use internally.

In [ ]:
model.zero_grad(set_to_none=True)

fobs_data, fobs_mask = F_obs.get_data(), F_obs.get_mask()
f_calc = model(hkl)
loss = (((f_calc.abs() - fobs_data) * fobs_mask) ** 2).mean()
loss.backward()

grad = model.xyz.refinable_params.grad
print(f'xyz grad shape: {tuple(grad.shape)},  ||grad||_2 = {grad.norm().item():.4e}')

AttributeError: 'NoneType' object has no attribute 'backward'

## Summary

| Approach | When to use | Returns |
|---|---|---|
| `model(hkl)` | Standard refinement, quick checks | complex `F_calc` |
| `FFT` class  | Need the density map, multiple HKL sets, change cell/spacegroup | density and/or `F_calc` |
| `torchref.base` functions | Custom density building, learned scattering factors, debugging | whatever you assemble |

All three are differentiable end-to-end and run on CPU or GPU.